# Test GEM Enviroment

This notebook runs a very simple Agent to play a number guessing game. The Agent uses binary search to gues sthe number between 1 and 10 (inclusive).

In [22]:
import re
import gem
from dspy import GEPA, LM
import dspy

In [31]:
# Config
import os


region = os.getenv("AWS_REGION")
token = os.getenv("AWS_BEARER_TOKEN_BEDROCK")
llama_1b = os.getenv("BEDROCK_LLAMA_1B")
llama_3b = os.getenv("BEDROCK_LLAMA_3B")
llama_8b = os.getenv("BEDROCK_LLAMA_8B")
gpt_oss_20b = os.getenv("BEDROCK_GPT_OSS_20B")
gpt_oss_120b = os.getenv("BEDROCK_GPT_OSS_120B")
huggingface_token = os.getenv("HF_TOKEN")

## Class Instantiation

In [24]:
class SimpleGuessAgent:
    def __init__(self) -> None:
        self.low: int | None = None
        self.high: int | None = None
        self.last_action: int | None = None

    def parse_range(self, observation: str) -> None:
        numbers = list(map(int, re.findall(r"\d+", observation)))
        if len(numbers) >= 2:
            self.low, self.high = numbers[0], numbers[1]

    def act(self, observation: str | None) -> str:
        # First step: read range
        if observation is not None and (self.low is None or self.high is None):
            self.parse_range(observation)

        # Update bounds from feedback
        if observation is not None and self.last_action is not None:
            if "higher" in observation:
                self.low = self.last_action + 1
            elif "lower" in observation:
                self.high = self.last_action - 1

        # Safety check
        if self.low is None or self.high is None:
            raise ValueError("Range not initialized from observation.")

        # Binary search guess
        self.last_action = (self.low + self.high) // 2
        return f"\\boxed{{{self.last_action}}}"

## Train the Agent

In [25]:
# Instantiate the GEM environment
env = gem.make("game:GuessTheNumber-v0-easy")
observation, info = env.reset()

print("Initial observation:", observation)

agent = SimpleGuessAgent()

turn = 1

while True:
    action = agent.act(observation)
    next_obs, reward, terminated, truncated, info = env.step(action)

    # Extract number from \boxed{}
    guess = action.replace("\\boxed{", "").replace("}", "")

    print(f"\nTurn {turn}")
    print("=" * 60)
    print(f"Agent guess: {guess}")
    print(f"Environment: {next_obs}")
    print(f"Reward: {reward}")

    observation = next_obs
    turn += 1

    if terminated or truncated:
        print("\nEpisode finished.")
        break

Initial observation: You are playing Guess The Number.
You have to guess the number between 1 and 10 (inclusive) within 4 turns.
As you enter your guess, the game will provide you with hints such as the target number is 'higher' or 'lower'.
You may provide your response in any manner. Only the number that is wrapped inside \boxed{} will be considered as your guess, for example, \boxed{5}.
As you play, the history of your guesses will be appended below. Use the information to complete the game before you run out of guesses.
Enter your first guess to start the game.


Turn 1
Agent guess: 5
Environment: At turn 1, you guessed 5, and the target number is higher than 5.
Reward: 0.0

Turn 2
Agent guess: 8
Environment: At turn 2, you guessed 8, and the target number is lower than 8.
Reward: 0.0

Turn 3
Agent guess: 6
Environment: Congratulations! You guessed the correct number 6 in 3 turns.
Reward: 1.0

Episode finished.


# DSPy + GEM

In [49]:
from dspy import Signature, InputField, OutputField, Module

# Configure DSPy to use your preferred LM (e.g., OpenAI, local model, etc.)
# Example with OpenAI:
# lm = dspy.OpenAI(model='gpt-4')
# dspy.settings.configure(lm=lm)

# Or with a local model via Ollama:
lm = LM(model=f"bedrock/{gpt_oss_20b}", temperature=0.7, max_tokens=8192)
dspy.settings.configure(lm=lm)

In [50]:
# 1. Define DSPy Signatures
class GuessSignature(Signature):
    """Generate a guess for the number guessing game based on environment feedback."""

    game_feedback = InputField(
        desc="The environment's response to previous guess (too high, too low, etc.)"
    )
    previous_guesses = InputField(desc="List of previous guesses and their feedback")
    min_range = InputField(desc="Current minimum possible number")
    max_range = InputField(desc="Current maximum possible number")

    guess = OutputField(desc="The next number to guess, wrapped in \\boxed{}")
    reasoning = OutputField(desc="Step-by-step reasoning for the guess")


class AnalysisSignature(Signature):
    """Analyze the game history to improve guessing strategy."""

    game_history = InputField(desc="Complete history of guesses and feedback")
    success = InputField(desc="Whether the game was successfully completed")
    turns_taken = InputField(desc="Number of turns taken")

    strategy_analysis = OutputField(
        desc="Analysis of what worked and what could be improved"
    )
    optimal_strategy = OutputField(desc="Improved strategy for future games")


# 2. Create DSPy Modules
class GuessGenerator(dspy.Module):
    """Module that generates the next guess."""

    def __init__(self):
        super().__init__()
        self.generate_guess = dspy.ChainOfThought(GuessSignature)

    def forward(self, game_feedback, previous_guesses, min_range, max_range):
        pred = self.generate_guess(
            game_feedback=game_feedback,
            previous_guesses=previous_guesses,
            min_range=min_range,
            max_range=max_range,
        )
        return dspy.Prediction(guess=pred.guess, reasoning=pred.reasoning)


class StrategyOptimizer(dspy.Module):
    """Module that analyzes and optimizes guessing strategy."""

    def __init__(self):
        super().__init__()
        self.analyze = dspy.Predict(AnalysisSignature)

    def forward(self, game_history, success, turns_taken):
        pred = self.analyze(
            game_history=game_history, success=success, turns_taken=turns_taken
        )
        return dspy.Prediction(
            strategy_analysis=pred.strategy_analysis,
            optimal_strategy=pred.optimal_strategy,
        )


# 3. Main DSPy Agent for GEM
class DSPyGEMAgent:
    def __init__(self, optimize=True):
        self.guess_generator = GuessGenerator()
        self.strategy_optimizer = StrategyOptimizer()
        self.game_history = []
        self.min_range = 1
        self.max_range = 100  # Default for easy mode
        self.previous_guesses = []
        self.optimize = optimize

        # Optional: Compile/optimize the modules with teleprompters
        if optimize:
            self.optimize_modules()

    def optimize_modules(self):
        """Use DSPy optimizers to improve the modules."""
        from dspy.teleprompt import BootstrapFewShot

        # Create a few examples for bootstrapping
        examples = [
            dspy.Example(
                game_feedback="Too low!",
                previous_guesses="[50: Too low]",
                min_range=51,
                max_range=100,
                guess="\\boxed{75}",
                reasoning="Binary search: midpoint between 51 and 100 is 75",
            ).with_inputs(
                "game_feedback", "previous_guesses", "min_range", "max_range"
            ),
            dspy.Example(
                game_feedback="Too high!",
                previous_guesses="[50: Too high]",
                min_range=1,
                max_range=49,
                guess="\\boxed{25}",
                reasoning="Binary search: midpoint between 1 and 49 is 25",
            ).with_inputs(
                "game_feedback", "previous_guesses", "min_range", "max_range"
            ),
        ]

        # Optimize the guess generator
        teleprompter = BootstrapFewShot(metric=self.validate_guess)
        self.guess_generator = teleprompter.compile(
            self.guess_generator, trainset=examples
        )

    def validate_guess(self, example, pred, trace=None):
        """Validation metric for guess quality."""
        try:
            # Check if guess is within range
            guess_num = int(pred.guess.replace("\\boxed{", "").replace("}", ""))
            if example.min_range <= guess_num <= example.max_range:
                return 1.0
        except:
            pass
        return 0.0

    def update_range(self, feedback, guess_num):
        """Update the possible number range based on feedback."""
        if "Too low" in feedback:
            self.min_range = max(self.min_range, guess_num + 1)
        elif "Too high" in feedback:
            self.max_range = min(self.max_range, guess_num - 1)
        # If correct, range becomes just that number

    def act(self, observation):
        """Generate the next action based on current observation."""

        # Parse observation
        if "Feedback:" in observation:
            feedback = observation.split("Feedback:")[1].strip()
        else:
            feedback = observation

        # Update range based on feedback from previous guess
        if self.previous_guesses and self.optimize:
            last_guess = self.previous_guesses[-1]
            try:
                last_guess_num = int(last_guess.split(":")[0].strip())
                self.update_range(feedback, last_guess_num)
            except:
                pass

        # Format previous guesses for the signature
        prev_guesses_str = ", ".join(
            [f"{g}" for g in self.previous_guesses[-5:]]
        )  # Last 5 guesses

        # Generate next guess using DSPy module
        result = self.guess_generator(
            game_feedback=feedback,
            previous_guesses=prev_guesses_str,
            min_range=str(self.min_range),
            max_range=str(self.max_range),
        )

        # Store reasoning for analysis
        print(f"Reasoning: {result.reasoning}")

        # Extract and store the guess
        guess = result.guess
        self.previous_guesses.append(f"{guess}: {feedback}")

        return guess

    def analyze_performance(self, success, turns_taken):
        """Analyze and improve strategy after each episode."""
        if not self.optimize:
            return

        game_history = "\n".join(
            [f"Turn {i+1}: {g}" for i, g in enumerate(self.previous_guesses)]
        )

        analysis = self.strategy_optimizer(
            game_history=game_history, success=success, turns_taken=turns_taken
        )

        print("\n" + "=" * 60)
        print("PERFORMANCE ANALYSIS")
        print("=" * 60)
        print(f"Strategy Analysis: {analysis.strategy_analysis}")
        print(f"Optimal Strategy: {analysis.optimal_strategy}")
        print("=" * 60)

        # Reset for next episode
        self.reset()

    def reset(self):
        """Reset agent state for new episode."""
        self.game_history = []
        self.min_range = 1
        self.max_range = 100
        self.previous_guesses = []


# 4. Training loop with optimization
def train_agent(num_episodes=10):
    """Train the DSPy agent on multiple episodes."""

    env = gem.make("game:GuessTheNumber-v0-easy")
    agent = DSPyGEMAgent(optimize=True)

    all_results = []

    for episode in range(num_episodes):
        print(f"\n{'='*60}")
        print(f"EPISODE {episode + 1}/{num_episodes}")
        print("=" * 60)

        observation, info = env.reset()
        print("Initial observation:", observation)

        turn = 1
        success = False

        while True:
            action = agent.act(observation)
            next_obs, reward, terminated, truncated, info = env.step(action)

            # Extract number from \boxed{}
            guess = action.replace("\\boxed{", "").replace("}", "")

            print(f"\nTurn {turn}")
            print("-" * 40)
            print(f"Agent guess: {guess}")
            print(f"Environment: {next_obs}")
            print(f"Reward: {reward}")

            observation = next_obs
            turn += 1

            if terminated or truncated:
                print("\nEpisode finished.")
                success = reward > 0  # Assuming positive reward means success
                break

        # Analyze performance after each episode
        agent.analyze_performance(success, turn - 1)
        all_results.append(
            {"episode": episode + 1, "turns": turn - 1, "success": success}
        )

    # Print summary
    print("\n" + "=" * 60)
    print("TRAINING SUMMARY")
    print("=" * 60)
    for result in all_results:
        status = "✓" if result["success"] else "✗"
        print(f"Episode {result['episode']}: {status} ({result['turns']} turns)")

    return agent, all_results


# 5. Run the training
if __name__ == "__main__":
    trained_agent, results = train_agent(num_episodes=5)

    # Optional: Save the optimized agent
    # dspy.save(trained_agent, "optimized_guess_agent.dspy")

100%|██████████| 2/2 [00:04<00:00,  2.02s/it]


Bootstrapped 2 full traces after 1 examples for up to 1 rounds, amounting to 2 attempts.

EPISODE 1/5
Initial observation: You are playing Guess The Number.
You have to guess the number between 1 and 10 (inclusive) within 4 turns.
As you enter your guess, the game will provide you with hints such as the target number is 'higher' or 'lower'.
You may provide your response in any manner. Only the number that is wrapped inside \boxed{} will be considered as your guess, for example, \boxed{10}.
As you play, the history of your guesses will be appended below. Use the information to complete the game before you run out of guesses.
Enter your first guess to start the game.

Reasoning: The target is between 1 and 10.  
Choosing the midpoint of the full range 1–10 gives 5, which is a balanced first guess.

Turn 1
----------------------------------------
Agent guess: 5
Environment: At turn 1, you guessed 5, and the target number is higher than 5.
Reward: 0.0
Reasoning: The game states we must gue